In [ ]:
## TASCA 1. Presentació del conjunt de dades

El conjunt de dades correspon a una clínica veterinària que recull informació sobre els serveis d'atenció a animals en diferents centres de Catalunya. 
L'objectiu de l'anàlisi és estudiar l'activitat veterinària, comparant els serveis, les espècies ateses, els centres, el nombre d'animals i la durada de les visites. 
També es vol detectar possibles errors o dades incompletes abans de fer l'anàlisi.

El fitxer conté 120 registres i les columnes següents:

- **servei**: tipus de servei veterinari realitzat(consulta, vacuna, radiografia, etc)
- **espècie**: espècie de l'animal atès(gos, gat, conill, ocells, exòtics...)
- **centre**: municipi o centre veterinari on s'ha realitzat l'atenció.
- **animals_atesos**: nombre d'animals atesos en el servei.
- **durada_min**: durada del servei en minuts.
- **data**: data en què es va realitzar l'atenció.

Les dades presenten alguns problemes de qualitat, com ara camps buits, espais sobrants, valors no numèrics i dates incorrectes. 
Per tant, abans de calcular estadístiques o crear gràfics, serà necessari netejar i validar el conjunt de dades.

In [ ]:
## Fases de l'anàlisi

1.	**Presentació del conjunt de dades:** descripció del sector, objectiu de l’anàlisi i explicació de les columnes.
2.	**Exploració inicial:** càrrega del CSV, visualització de les primeres files, dimensions del conjunt de dades i tipus de dades.
3.	**Detecció d'incidències:** identificació de camps obligatoris buits, valors numèrics incorrectes, dates invàlides i inconsistències de format.
4.	**Neteja i validació:** aplicació de les transformacions necessàries i definició de les regles per determinar quins registres són vàlids.
5.	**Gestió dels registres incorrectes:** identificació dels registres rebutjats i explicació dels motius pels quals no es poden acceptar.
6.	**Anàlisi de les dades netes:** càlcul d’estadístiques i indicadors rellevants, amb agrupacions per les variables de classificació quan sigui convenient.
7.	**Resultats i conclusions:** resum dels principals resultats i valoració de la qualitat de les dades


In [ ]:
# EXERCICI 1 - Exploració inicial

import csv
from datetime import datetime

nom_fitxer = "clinica_veterinaria.csv"

with open(nom_fitxer, mode="r", encoding="utf-8-sig", newline="") as fitxer: #-sig elimina el BOM
    lector = csv.DictReader(fitxer)
    camps = lector.fieldnames
    registres = list(lector)



print("Camps del CSV:", camps)
print("Nombre total de registres:", len(registres))
if registres:
    print("Exemple de primer registre:", registres[0])

In [ ]:
## EXERCICI 2 - Detecció d'incidències

comptador_problemes = {
    "camps_buit": 0,
    "animals_invalids": 0,
    "durada_invalides": 0,
    "dates_invalides": 0
}

def es_numero_positiu(valor, tipus=float):
    """Auxiliar per validar números positius de manera senzilla."""
    try:
        return tipus(str(valor).replace(",", ".")) > 0
    except (ValueError, TypeError):
        return False

for reg in registres:

    # Camps obligatoris buits
    camps_obligatoris = ["servei", "especie", "centre"]
    te_camp_buit = False
    for camp in camps_obligatoris:
            val = str(reg.get(camp, "")).strip().lower()
            if not val or val == "nan":
                te_camp_buit = True
                break
    if te_camp_buit:
            comptador_problemes["camps_buit"] += 1

    # Animals atesos: enter positiu
    if not es_numero_positiu(reg.get("animals_atesos"), int):
        comptador_problemes["animals_invalids"] += 1

    # Durada: nombre positiu
    if not es_numero_positiu(reg.get("durada_min")):
            comptador_problemes["durada_invalides"] += 1

    # Data amb format YYYY-MM-DD
    data_text = str(reg.get("data", "")).strip()
    try:
        datetime.strptime(data_text, "%Y-%m-%d")
    except (ValueError, TypeError):
        comptador_problemes["dates_invalides"] += 1

print("Resum d'incidències detectades:")

for tipus, quantitat in comptador_problemes.items():
    print(f"- {tipus}: {quantitat}")



In [ ]:
# EXERCICI 3 - Neteja i validació:

import csv
from datetime import datetime

nom_fitxer = "clinica_veterinaria.csv"

registres_nets = []
registres_rebutjats = []


def neteja_text(valor):
    """Converteix un valor a text i elimina els espais sobrants."""
    if valor is None:
        return ""
    return str(valor).strip()


with open(nom_fitxer, mode="r", encoding="utf-8-sig", newline="") as fitxer:
    lector = csv.DictReader(fitxer)
    
    for numero_linia, reg in enumerate(lector, start=2):
        errors = []

        # 1. Neteja i validació dels camps obligatoris
        servei = neteja_text(reg.get("servei"))
        especie = neteja_text(reg.get("especie"))
        centre = neteja_text(reg.get("centre"))

        if not servei:
            errors.append("servei buit")
        if not especie:
            errors.append("especie buida")
        if not centre:
            errors.append("centre buit")

        # 2. Conversió d'animals_atesos a enter
        animals_text = neteja_text(reg.get("animals_atesos"))
        try:
            animals_atesos = int(animals_text)
            if animals_atesos <= 0:
                errors.append("No és positiu")
                animals_atesos = None
        except (ValueError, TypeError):
            animals_atesos = None
            errors.append("No és un enter vàlid")

        # 3. Conversió de durada_min a nombre decimal
        durada_text = neteja_text(reg.get("durada_min")).replace(",", ".")
        try:
            durada_min = float(durada_text)
            if durada_min <= 0:
                errors.append("No és positiva")
                durada_min = None
        except (ValueError, TypeError):
            durada_min = None
            errors.append("No és un nombre vàlid")


        # 4. Validació i normalització de la data
        data_text = neteja_text(reg.get("data"))
        try:
            data = datetime.strptime(data_text, "%Y-%m-%d")
            data_normalitzada = data.strftime("%Y-%m-%d")
        except (ValueError, TypeError):
            data_normalitzada = None
            errors.append("data no vàlida")

        # 5. Classificació final del registre
        if not errors:
            registre_net = {
                "servei": servei,
                "especie": especie,
                "centre": centre,
                "animals_atesos": animals_atesos,
                "durada_min": durada_min,
                "data": data_normalitzada
            }
            registres_nets.append(registre_net)
        else:
            registre_rebutjat = reg.copy()
            registre_rebutjat["linia"] = numero_linia
            registre_rebutjat["errors"] = "; ".join(errors)
            registres_rebutjats.append(registre_rebutjat)


print(f"Vàlids: {len(registres_nets)}")
print(f"Rebutjats: {len(registres_rebutjats)}")

In [ ]:
## EXERCICI 4 - Gestió de registres incorrectes

import csv
from datetime import datetime

nom_fitxer = "clinica_veterinaria.csv"

print(f"Total de registres rebutjats: {len(registres_rebutjats)}")
print()

for registre in registres_rebutjats:
    print(f"Línia {registre['linia']}: {registre['errors']}")

In [ ]:
## EXERCICI 5 - Anàlisi de dades netes

import csv
from datetime import datetime

nom_fitxer = "clinica_veterinaria.csv"



print("=" * 25)
print("ANÀLISI DE LES DADES NETES")
print("=" * 25)

# Nombre total de registres vàlids
print(f"Nombre de registres: {len(registres_nets)}")

# Estadístiques generals
animals_totals = sum(
    registre["animals_atesos"]
    for registre in registres_nets
)

durades = [
    registre["durada_min"]
    for registre in registres_nets
]

print(f"Animals atesos en total: {animals_totals}")
print(f"Durada mínima: {min(durades):.2f} minuts")
print(f"Durada màxima: {max(durades):.2f} minuts")



In [32]:
## EXECICI 6 - Resultats i Conclusions

import csv
from datetime import datetime

nom_fitxer = "clinica_veterinaria.csv"

print("=" * 25)
print("RESULTATS")
print("=" * 25)

print(f"S'han processat 120 registres totals, dels quals s'han acceptat {len(registres_nets)} i rebutjat {len(registres_rebutjats)}.")

print("=" * 25)
print("CONCLUSIONS")
print("=" * 25)
print("Qualitat de les dades: Les incidències més comunes han estat camps obligatoris buits, valors numèrics no positius i errors en el format de les dates.")
print("Valoració final: Fitxer net, dades fiables per analitzar activitat a la clíniques veterinaries.")

RESULTATS
S'han processat 120 registres totals, dels quals s'han acceptat 90 i rebutjat 30.
CONCLUSIONS
Qualitat de les dades: Les incidències més comunes han estat camps obligatoris buits, valors numèrics no positius i errors en el format de les dates.
Valoració final: Fitxer net, dades fiables per analitzar activitat a la clíniques veterinaries.
